In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

endpoints = w.serving_endpoints.list()

print("=== Available AI Model Serving Endpoints ===\n")
for ep in endpoints:
    status = "READY" if ep.state and ep.state.ready else "NOT READY"
    if ep.config and ep.config.served_entities:
        for se in ep.config.served_entities:
            model_name = se.entity_name
            model_type = "External" if se.external_model else "Foundation"
            provider = se.external_model.provider if se.external_model else "Databricks"
            print(f"Endpoint: {ep.name}")
            print(f"  Model: {model_name}")
            print(f"  Type: {model_type} | Provider: {provider} | Status: {status}")
            print()

In [0]:
%sql
SELECT ai_query(
  'databricks-gpt-oss-20b',
  'Explain in one short paragraph what the RP2040 microcontroller is.'
) AS response;


In [0]:
%sql
WITH retrieved AS (
  SELECT
    file_name,
    page_number,
    product,
    document_type,
    text,
    search_score
  FROM vector_search(
    index => 'microcontrollers.rag_project.document_search',
    query_text => 'How much SRAM does RP2040 have?',
    num_results => 5
  )
)
SELECT
  CONCAT(
    'SOURCE: ', file_name,
    ' | PAGE: ', page_number,
    ' | PRODUCT: ', product,
    ' | DOCUMENT TYPE: ', document_type,
    '\n',
    text
  ) AS context
FROM retrieved;


In [0]:
%sql
WITH retrieved AS (
  SELECT
    file_name,
    page_number,
    product,
    document_type,
    text,
    search_score
  FROM vector_search(
    index => 'microcontrollers.rag_project.document_search',
    query_text => 'How much SRAM does RP2040 have?',
    num_results => 5
  )
),

context AS (
  SELECT
    array_join(
      collect_list(
        CONCAT(
          'SOURCE: ', file_name,
          ' | PAGE: ', CAST(page_number AS STRING),
          ' | PRODUCT: ', product,
          ' | DOCUMENT TYPE: ', document_type,
          '\n',
          text
        )
      ),
      '\n\n---\n\n'
    ) AS context_text
  FROM retrieved
)

SELECT
  ai_query(
    'databricks-gpt-oss-20b',
    CONCAT(
      'You are a technical documentation assistant.

',
      'Answer the user question using ONLY the retrieved documentation below.

',
      'Rules:
',
      '1. Do not use outside knowledge.
',
      '2. Do not invent or infer facts that are not supported by the retrieved documentation.
',
      '3. If the retrieved documentation does not contain enough information to answer the question, say so.
',
      '4. Cite the source PDF filename and page number for factual claims.
',
      '5. Keep the answer concise and technically precise.

',
      'USER QUESTION:
',
      'How much SRAM does RP2040 have?

',
      'RETRIEVED DOCUMENTATION:
',
      context_text
    )
  ) AS response
FROM context;


In [0]:
%sql
WITH params AS (
  SELECT 'What security features does RP2350 provide?' AS question
),

retrieved AS (
  SELECT *
  FROM vector_search(
    index => 'microcontrollers.rag_project.document_search',
    query_text => (SELECT question FROM params),
    num_results => 5
  )
),

context AS (
  SELECT
    (SELECT question FROM params) AS question,
    array_join(
      collect_list(
        CONCAT(
          'SOURCE: ', file_name,
          ' | PAGE: ', CAST(page_number AS STRING),
          ' | PRODUCT: ', product,
          ' | DOCUMENT TYPE: ', document_type,
          '\n',
          text
        )
      ),
      '\n\n---\n\n'
    ) AS context_text
  FROM retrieved
)

SELECT
  ai_query(
    'databricks-gpt-oss-20b',
    CONCAT(
      'You are a technical documentation assistant.

',
      'Answer the user question using ONLY the retrieved documentation below.

',
      'Rules:
',
      '1. Do not use outside knowledge.
',
      '2. Do not invent facts.
',
      '3. If the documentation does not contain enough information, say so.
',
      '4. Cite the source PDF filename and page number for factual claims.
',
      '5. Keep the answer concise and technically precise.

',
      'USER QUESTION:
',
      question,
      '

RETRIEVED DOCUMENTATION:
',
      context_text
    )
  ) AS response
FROM context;


In [0]:
%sql
WITH retrieved AS (
  SELECT
    file_name,
    page_number,
    product,
    document_type,
    text,
    search_score
  FROM vector_search(
    index => 'microcontrollers.rag_project.document_search',
    query_text => 'What security features does RP2350 provide?',
    num_results => 5
  )
),

context AS (
  SELECT
    array_join(
      collect_list(
        CONCAT(
          'SOURCE: ', file_name,
          ' | PAGE: ', CAST(page_number AS STRING),
          ' | PRODUCT: ', product,
          ' | DOCUMENT TYPE: ', document_type,
          '\n',
          text
        )
      ),
      '\n\n---\n\n'
    ) AS context_text
  FROM retrieved
)

SELECT
  ai_query(
    'databricks-gpt-oss-20b',
    CONCAT(
      'You are a technical documentation RAG assistant.

',
      'IMPORTANT: The retrieved documentation below is the ONLY source of truth.

',
      'Grounding rules:
',
      '- Use only information explicitly stated in the retrieved documentation.
',
      '- Do not use your pretrained knowledge or outside information.
',
      '- Do not add examples, specifications, explanations, or interpretations that are not explicitly present in the retrieved text.
',
      '- Do not infer missing details.
',
      '- If the retrieved documentation does not provide enough information, explicitly say that the information is not available in the retrieved context.
',
      '- Every factual claim must include a citation in this exact format: [Source: filename, p. X].
',
      '- Keep the answer concise and technically precise.

',
      'USER QUESTION:
',
      'What security features does RP2350 provide?

',
      'RETRIEVED DOCUMENTATION:
',
      context_text
    )
  ) AS response
FROM context;


In [0]:
%sql
SELECT
  file_name,
  page_number,
  product,
  document_type,
  chunk_id,
  search_score
FROM vector_search(
  index => 'microcontrollers.rag_project.document_search',
  query_text => 'What security features does RP2350 provide?',
  num_results => 5
);


In [0]:
%sql
SELECT
    file_name,
    page_number,
    product,
    document_type,
    chunk_id,
    text,
    search_score
FROM VECTOR_SEARCH(
    index => 'microcontrollers.rag_project.document_search',
    query_text => 'What security features does RP2350 provide?',
    num_results => 5
)
ORDER BY search_score DESC;


In [0]:
%sql

DECLARE OR REPLACE VARIABLE user_question STRING
DEFAULT 'What security features does RP2350 provide?';

WITH retrieved AS (
    SELECT
        file_name,
        page_number,
        product,
        document_type,
        chunk_id,
        text,
        search_score
    FROM VECTOR_SEARCH(
        index => 'microcontrollers.rag_project.document_search',
        query_text => user_question,
        num_results => 5
    )
),

context AS (
    SELECT
        concat_ws(
            '\n\n',
            collect_list(
                concat(
                    'SOURCE: ', file_name,
                    ' | PAGE: ', CAST(page_number AS STRING),
                    ' | PRODUCT: ', product,
                    ' | DOCUMENT TYPE: ', document_type,
                    '\n',
                    text
                )
            )
        ) AS retrieved_context
    FROM retrieved
)

SELECT
    ai_query(
        'databricks-gpt-oss-20b',
        concat(
            'You are a technical documentation assistant for RP2040 and RP2350. ',
            'Answer the user question using ONLY the retrieved documentation provided below. ',
            'Do not use outside knowledge. ',
            'Do not invent specifications, features, numbers, or explanations. ',
            'If the retrieved documentation does not contain enough information, ',
            'clearly say that the provided documentation does not contain enough information. ',
            'For factual claims, cite the source using this format: ',
            '[Source: filename, page N]. ',
            'Use the page number provided in the retrieved context. ',
            '\n\nUSER QUESTION:\n',
            user_question,
            '\n\nRETRIEVED DOCUMENTATION:\n',
            retrieved_context
        )
    ) AS response
FROM context;


In [0]:
%sql

DECLARE OR REPLACE VARIABLE user_question STRING
DEFAULT 'How much SRAM does RP2040 have?';

WITH retrieved AS (
    SELECT
        file_name,
        page_number,
        product,
        document_type,
        chunk_id,
        text,
        search_score
    FROM VECTOR_SEARCH(
        index => 'microcontrollers.rag_project.document_search',
        query_text => user_question,
        num_results => 5
    )
),

context AS (
    SELECT
        concat_ws(
            '\n\n',
            collect_list(
                concat(
                    'SOURCE: ', file_name,
                    ' | PAGE: ', CAST(page_number AS STRING),
                    ' | PRODUCT: ', product,
                    ' | DOCUMENT TYPE: ', document_type,
                    '\n',
                    text
                )
            )
        ) AS retrieved_context
    FROM retrieved
)

SELECT
    ai_query(
        'databricks-gpt-oss-20b',
        concat(
            'You are a technical documentation assistant for RP2040 and RP2350. ',
            'Answer the user question using ONLY the retrieved documentation provided below. ',
            'Do not use outside knowledge. ',
            'Do not invent specifications, features, numbers, or explanations. ',
            'If the retrieved documentation does not contain enough information, ',
            'clearly say that the provided documentation does not contain enough information. ',
            'For factual claims, cite the source using this format: ',
            '[Source: filename, page N]. ',
            'Use the page number provided in the retrieved context. ',
            '\n\nUSER QUESTION:\n',
            user_question,
            '\n\nRETRIEVED DOCUMENTATION:\n',
            retrieved_context
        )
    ) AS response
FROM context;


In [0]:
%sql

DECLARE OR REPLACE VARIABLE user_question STRING
DEFAULT 'What security features does RP2350 provide?';

WITH retrieved AS (
    SELECT
        file_name,
        page_number,
        product,
        document_type,
        chunk_id,
        text,
        search_score
    FROM VECTOR_SEARCH(
        index => 'microcontrollers.rag_project.document_search',
        query_text => user_question,
        num_results => 5
    )
),

context AS (
    SELECT
        concat_ws(
            '\n\n--- DOCUMENT CHUNK ---\n\n',
            collect_list(
                concat(
                    'SOURCE FILE: ', file_name,
                    '\nPAGE: ', CAST(page_number AS STRING),
                    '\nPRODUCT: ', product,
                    '\nDOCUMENT TYPE: ', document_type,
                    '\nCHUNK ID: ', chunk_id,
                    '\n\nTEXT:\n',
                    text
                )
            )
        ) AS retrieved_context
    FROM retrieved
)

SELECT
    ai_query(
        'databricks-gpt-oss-20b',
        concat(
            'You are a technical documentation assistant for RP2040 and RP2350. ',
            '\n\n',
            'STRICT GROUNDING RULES:',
            '\n1. Answer using ONLY information explicitly stated in the retrieved documentation.',
            '\n2. Do not use your general or pretrained knowledge.',
            '\n3. Do not infer, extrapolate, or fill in missing technical details.',
            '\n4. Do not introduce technical terms, specifications, algorithms, hardware blocks, ',
            'or capabilities unless they are explicitly supported by the retrieved text.',
            '\n5. Every factual claim must be supported by one or more retrieved document chunks.',
            '\n6. Cite the source file and page that support each factual claim.',
            '\n7. If the retrieved documentation does not contain enough information, say so clearly.',
            '\n8. Do not cite a page merely because it is related to the topic; the cited page must support the claim.',
            '\n\n',
            'CITATION FORMAT:',
            '\nUse exactly this format:',
            '[Source: filename, page N]',
            '\n\n',
            'USER QUESTION:',
            '\n',
            user_question,
            '\n\n',
            'RETRIEVED DOCUMENTATION:',
            '\n',
            retrieved_context
        )
    ) AS response
FROM context;


In [0]:
%sql

DECLARE OR REPLACE VARIABLE user_question STRING
DEFAULT 'How much SRAM does RP2040 have?';

WITH retrieved AS (
    SELECT
        file_name,
        page_number,
        product,
        document_type,
        chunk_id,
        text,
        search_score
    FROM VECTOR_SEARCH(
        index => 'microcontrollers.rag_project.document_search',
        query_text => user_question,
        num_results => 5
    )
),

context AS (
    SELECT
        concat_ws(
            '\n\n--- DOCUMENT CHUNK ---\n\n',
            collect_list(
                concat(
                    'SOURCE FILE: ', file_name,
                    '\nPAGE: ', CAST(page_number AS STRING),
                    '\nPRODUCT: ', product,
                    '\nDOCUMENT TYPE: ', document_type,
                    '\nCHUNK ID: ', chunk_id,
                    '\n\nTEXT:\n',
                    text
                )
            )
        ) AS retrieved_context
    FROM retrieved
)

SELECT
    ai_query(
        'databricks-gpt-oss-20b',
        concat(
            'You are a technical documentation assistant for RP2040 and RP2350. ',
            '\n\n',
            'STRICT GROUNDING RULES:',
            '\n1. Answer using ONLY information explicitly stated in the retrieved documentation.',
            '\n2. Do not use your general or pretrained knowledge.',
            '\n3. Do not infer, extrapolate, or fill in missing technical details.',
            '\n4. Do not introduce technical terms, specifications, algorithms, hardware blocks, ',
            'or capabilities unless they are explicitly supported by the retrieved text.',
            '\n5. Every factual claim must be supported by one or more retrieved document chunks.',
            '\n6. Cite the source file and page that support each factual claim.',
            '\n7. If the retrieved documentation does not contain enough information, say so clearly.',
            '\n8. Do not cite a page merely because it is related to the topic; the cited page must support the claim.',
            '\n\n',
            'CITATION FORMAT:',
            '\nUse exactly this format:',
            '[Source: filename, page N]',
            '\n\n',
            'USER QUESTION:',
            '\n',
            user_question,
            '\n\n',
            'RETRIEVED DOCUMENTATION:',
            '\n',
            retrieved_context
        )
    ) AS response
FROM context;


In [0]:
%sql

DECLARE OR REPLACE VARIABLE user_question STRING
DEFAULT 'What are the main differences between RP2040 and RP2350?';

WITH retrieved AS (
    SELECT
        file_name,
        page_number,
        product,
        document_type,
        chunk_id,
        text,
        search_score
    FROM VECTOR_SEARCH(
        index => 'microcontrollers.rag_project.document_search',
        query_text => user_question,
        num_results => 5
    )
),

context AS (
    SELECT
        concat_ws(
            '\n\n--- DOCUMENT CHUNK ---\n\n',
            collect_list(
                concat(
                    'SOURCE FILE: ', file_name,
                    '\nPAGE: ', CAST(page_number AS STRING),
                    '\nPRODUCT: ', product,
                    '\nDOCUMENT TYPE: ', document_type,
                    '\nCHUNK ID: ', chunk_id,
                    '\n\nTEXT:\n',
                    text
                )
            )
        ) AS retrieved_context
    FROM retrieved
),

answer AS (
    SELECT
        ai_query(
            'databricks-gpt-oss-20b',
            concat(
                'You are a technical documentation assistant for RP2040 and RP2350.',
                '\n\nSTRICT GROUNDING RULES:',
                '\n1. Answer using ONLY information explicitly stated in the retrieved documentation.',
                '\n2. Do not use general or pretrained knowledge.',
                '\n3. Do not infer or extrapolate missing technical details.',
                '\n4. Do not introduce specifications or capabilities unless supported by the retrieved text.',
                '\n5. Every factual claim must be supported by retrieved documentation.',
                '\n6. Cite the source file and page supporting each factual claim.',
                '\n7. If the documentation is insufficient, say so.',
                '\n8. Do not cite a page unless it supports the claim.',
                '\n\nCITATION FORMAT:',
                '\n[Source: filename, page N]',
                '\n\nUSER QUESTION:',
                '\n',
                user_question,
                '\n\nRETRIEVED DOCUMENTATION:',
                '\n',
                retrieved_context
            )
        ) AS response
    FROM context
)

SELECT
    response
FROM answer;


In [0]:
%sql
WITH retrieved AS (
    SELECT
        file_name,
        page_number,
        product,
        document_type,
        chunk_id,
        text,
        search_score
    FROM VECTOR_SEARCH(
        index => 'microcontrollers.rag_project.document_search',
        query_text => 'YOUR QUESTION',
        num_results => 5
    )
),

context AS (
    SELECT
        concat_ws(
            '\n\n',
            collect_list(
                concat(
                    'Source: ', file_name,
                    '\nPage: ', page_number,
                    '\nProduct: ', product,
                    '\nContent: ', text
                )
            )
        ) AS retrieved_context
    FROM retrieved
)

SELECT ai_query(
    'databricks-gpt-oss-20b',
    concat(
        'You are a documentation question-answering assistant.

Answer ONLY using the retrieved documentation.

Rules:
1. Do not use outside knowledge.
2. Do not invent facts.
3. Do not infer specifications that are not explicitly stated.
4. Cite every factual claim using document name and page number.
5. If the answer is not supported by the retrieved documentation, say:
   "The retrieved documentation does not contain enough information to answer this question."

User question:
',
        'How much SRAM does RP2040 have?',
        '

Retrieved documentation:
',
        retrieved_context
    )
) AS response
FROM context;


In [0]:
%sql

WITH retrieved AS (
    SELECT
        file_name,
        page_number,
        product,
        document_type,
        chunk_id,
        text,
        search_score
    FROM VECTOR_SEARCH(
        index => 'microcontrollers.rag_project.document_search',
        query_text => 'How much SRAM does RP2040 have?',
        num_results => 5
    )
),

context AS (
    SELECT
        concat_ws(
            '\n\n',
            collect_list(
                concat(
                    'Source: ', file_name,
                    '\nPage: ', CAST(page_number AS STRING),
                    '\nProduct: ', product,
                    '\nDocument type: ', document_type,
                    '\nContent: ', text
                )
            )
        ) AS retrieved_context
    FROM retrieved
)

SELECT ai_query(
    'databricks-gpt-oss-20b',
    concat(
        'You are a documentation question-answering assistant.

Answer the user question using ONLY the retrieved documentation.

Rules:
1. Use only information explicitly supported by the retrieved documentation.
2. Do not use your general knowledge.
3. Do not make assumptions or infer missing specifications.
4. If the retrieved documentation does not contain enough information to answer the question, say:
"The retrieved documentation does not contain enough information to answer this question."
5. Every factual claim should include the source document name and page number.
6. Do not invent citations.
7. Keep the answer concise and directly answer the user question.

User question:
How much SRAM does RP2040 have?

Retrieved documentation:
',
        retrieved_context
    )
) AS response
FROM context;
